# DEDUPLICACIÓN DE PRODUCTOS - PRUEBA 1.0

In [6]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import dedupe
import warnings
import csv
print("✅ Librerías cargadas")
print(f"📊 Pandas versión: {pd.__version__}")
print(f"🔄 Dedupe instalado correctamente")

✅ Librerías cargadas
📊 Pandas versión: 2.3.3
🔄 Dedupe instalado correctamente


In [7]:
notebook_dir = os.path.dirname(os.path.abspath('__file__'))
data_odoo_path = os.path.join(notebook_dir, '..', 'data', 'raw', 'ProductosOdoo.csv')
df_odoo = pd.read_csv(data_odoo_path)
print(f"📦 Total productos: {len(df_odoo)}")
print(f"📋 Columnas: {list(df_odoo.columns)}")
df_odoo.head(5)

📦 Total productos: 14215
📋 Columnas: ['id', 'name', 'barcode', 'default_code', 'x_studio_many2many_field_37q_1irl4uc58', 'categ_id', 'seller_ids']


,id,name,barcode,default_code,x_studio_many2many_field_37q_1irl4uc58,categ_id,seller_ids
0,__export__.product_template_31022_88373f19,ADORNO HALLOWEN,7453066213137,CF00614/autoriza noris,NaN,Cumpleaños / Artículos de Fiesta,NaN
1,__export__.product_template_219315_4256fd8c,ARCHIVADOR VERTICAL 3 GAVETAS Y CERRADURA,NaN,NaN,NaN,All,NaN
2,__export__.product_template_219316_dfced89f,ARCHIVADOR VERTICAL 4 GAVETAS Y CERRADURA,NaN,NaN,NaN,All,NaN
3,__export__.product_template_76612_602c2239,ARTICULOS DE NAVIDAD,93,9,NaN,Navidad / Bolas,NaN
4,__export__.product_template_29991_44f2fe53,ARTICULOS FIESTA,17,1,NaN,Cumpleaños / Artículos de Fiesta,NaN


In [8]:
data_inter_path = os.path.join(notebook_dir, '..', 'data', 'raw', 'ProductosInterfuerza.xlsx')
df_int = pd.read_excel(data_inter_path)
print(f"📦 Total productos: {len(df_int)}")
print(f"📋 Columnas: {list(df_int.columns)}")
df_int.head(5)

📦 Total productos: 73681
📋 Columnas: ['Id', 'UPC Code', 'Ubicacion', 'Item Number', 'Tipo', 'Nombre', 'Proveedor Principal', 'Marca', 'Pais de Origen', 'Punto de ReOrden', 'Ultimo Precio Proveedor', 'InStock', 'Costo Promedio', 'Precio de Lista Predeterminada', 'Costo Promedio WH', 'Ultimo Precio Proveedor Purchase', 'Ultimo Costo Entrada', 'Ultimo Costo Unitario F', 'Ultimo Costo Unitario', 'Margen ', 'Creacion', 'Actualizado', 'Categoria L1', 'Categoria L2', 'Categoria L3', 'Agente', 'Status', 'Cuenta de Venta', 'Cuenta Costo', 'Cuenta de Inventario', 'Imagen', 'Comision', 'Grosor', 'Ancho', 'Altura', 'Largo', 'Impuesto', 'Lote', 'Expiracion', 'Color', 'Talla', 'Temporada', 'Prod Madre', 'Matriz Padre', 'Matriz Hijo', 'Arancel', 'Material', 'UoM InStock', 'UoM Compra', 'UoM Venta', 'Tags', 'Peso', 'Detalle']


,Id,UPC Code,Ubicacion,Item Number,Tipo,Nombre,Proveedor Principal,Marca,Pais de Origen,Punto de ReOrden,...,Matriz Padre,Matriz Hijo,Arancel,Material,UoM InStock,UoM Compra,UoM Venta,Tags,Peso,Detalle
0,PRO101109,7512155423106,BODEGA PRINCIPAL,A404/6917,PRODUCTO,GORRA FBI 7512155423106,OMER SA,GENERICO,NaN,0,...,NO,NO,NaN,NaN,NaN,NaN,NaN,NaN,0.0,SOMBRERO SHERIFF
1,PRO104891,40568000067,NaN,405686/25.95OFERTA5.00,PRODUCTO,VESTIDO LARGO DAMA CHINO,CAPI WORLDWIDE SA,ZZZZZ,NaN,0,...,NO,NO,NaN,NaN,NaN,NaN,NaN,NaN,0.0,VESTIDO LARGO DAMA CHINO
2,PRO107818,7465376497664,NaN,9599/HB2473-18,PRODUCTO,BIRRETE GRADUACION NEGRO-AZUL TELA 7465376497664,OMER SA,GENERICO,NaN,0,...,NO,NO,NaN,NaN,NaN,NaN,NaN,NaN,0.0,VIRRETE GRADUACION TELA
3,PRO107822,6930180607994,NaN,AFD-799,PRODUCTO,NUMEROS LED CHICO AFD-799,SOLARTE DE PANAMA SA,FIESTAS DAISY,NaN,0,...,NO,NO,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NUMEROS LED CHICO AFD-799
4,PRO107852,60000001438,BODEGA PRINCIPAL,601430,PRODUCTO,ARREGLO GRADUACION GLOBOS,CORPORACION DAISY SA,DAISY,NaN,0,...,NO,NO,NaN,NaN,NaN,NaN,NaN,NaN,0.0,ARREGLO GRADUACION GLOBOS


In [9]:
# ============================================
# Diagnóstico de productos con o sin marcas
# ============================================

df_copy = df_odoo.copy()
# Productos nulos o vacíos
df_null = df_copy['x_studio_many2many_field_37q_1irl4uc58'].isna()
df_whitespace = df_copy['x_studio_many2many_field_37q_1irl4uc58'].astype(str).str.contains(r'^\s*$', na=False)
df_empty = df_null | df_whitespace

total_empties = df_empty.sum() 
with_brands = len(df_copy) - total_empties 
# Conteo de valores de la columna Marca que no están vacíos
brands = df_copy[~df_empty]['x_studio_many2many_field_37q_1irl4uc58'].value_counts()

print(f"📊 DIAGNÓSTICO DE MARCAS:")
print(f"   Total productos: {len(df_copy) }")
print(f"   Con marca: {with_brands} ({with_brands/len(df_copy) *100:.2f}%)")
print(f"   Sin marca: {total_empties} ({total_empties/len(df_copy) *100:.2f}%)")

print(f"\n📊 ESTADÍSTICAS DE MARCAS:")
print(f"   Total de marcas únicas: {len(brands)}")
print(f"   Marca más frecuente: {brands.index[0]} ({brands.iloc[0]} productos)")
print(f"   Marca menos frecuente: {brands.index[-1]} ({brands.iloc[-1]} productos)")

# Marcas que aparecen solo una vez
one_register_brands = brands[brands == 1]
print(f"   Marcas con un solo producto: {len(one_register_brands)} ({len(one_register_brands)/len(brands)*100:.2f}% del total de marcas)")
# print(marcas_una_vez.head(50))


📊 DIAGNÓSTICO DE MARCAS:
   Total productos: 14215
   Con marca: 7261 (51.08%)
   Sin marca: 6954 (48.92%)

📊 ESTADÍSTICAS DE MARCAS:
   Total de marcas únicas: 616
   Marca más frecuente: POINTER (690 productos)
   Marca menos frecuente: THIRD CLASSROOM (1 productos)
   Marcas con un solo producto: 292 (47.40% del total de marcas)


In [10]:
""" 
Normalización de códigos de barra
Convierte código de barras a string y elimina ceros al inicio
"""
def barcode_normalization(code):
    if pd.isna(code): # Evalua si está vacío
        return None
    code = str(code).strip()
    try:
        # Esto elimina ceros al inicio automáticamente
        code = str(int(float(code)))
    except:
        pass
    return code

df_odoo['barcode_norm'] = df_odoo['barcode'].apply(barcode_normalization)
df_int['barcode_norm'] = df_int['UPC Code'].apply(barcode_normalization)

print("🔍 Ejemplos de normalización:")
overview = pd.DataFrame({
    'original_odoo': df_odoo['barcode'].head(5),
    'normalizado_odoo': df_odoo['barcode_norm'].head(5),
    'original_int': df_int['UPC Code'].head(5),
    'normalizado_int': df_int['barcode_norm'].head(5)
})
print(overview)

🔍 Ejemplos de normalización:
   original_odoo normalizado_odoo   original_int normalizado_int
0  7453066213137    7453066213137  7512155423106   7512155423106
1            NaN             None    40568000067     40568000067
2            NaN             None  7465376497664   7465376497664
3             93               93  6930180607994   6930180607994
4             17               17    60000001438     60000001438


In [11]:
# Crear diccionario de Interfuerza: barcode_norm → Marca, tomando registros que tengan ambos campos
# dropna() elimina cualquier fila que tenga al menos un valor nulo
# subset selecciona una porción específica de un Dataframe
df_int_valids = df_int.dropna(subset=['barcode_norm', 'Marca']) 

# Eliminar duplicados de barcode normalizado
df_int_uniques = df_int_valids.drop_duplicates(subset=['barcode_norm'])

# Creación de diccionario
# Unión de 'barcode_norm' con 'Marca' en tuplas usando zip()
brands_map = dict(zip(df_int_uniques['barcode_norm'], df_int_uniques['Marca'])) 
print(f"✅ Mapa creado con {len(brands_map)} códigos de barras únicos")

✅ Mapa creado con 73303 códigos de barras únicos


In [12]:
# ================================================
# Identificación de códigos con múltiples marcas
# ================================================
# Agrupa los códigos df_int_valids mostrando los valores de marca por cada código
# nunique() Muestra la cantidad de marcas únicas por cada código
conflicts_per_code = df_int_valids.groupby('barcode_norm')['Marca'].nunique()
# Verificación si hay códigos con más de una marca
codes_with_conflicts = conflicts_per_code[conflicts_per_code > 1]

print("="*61)
print("INFORME DE CALIDAD \nCONFLICTOS DE MARCAS POR CÓDIGO DE BARRAS")
print("="*61)

print(f"\n ESTADÍSTICAS GENERALES:")
print(f"   Total registros en Interfuerza: {len(df_int_valids):,}")
print(f"   Total códigos de barras únicos: {df_int_valids['barcode_norm'].nunique():,}")
print(f"   Códigos con múltiples marcas: {len(codes_with_conflicts):,}")
print(f"   Porcentaje del total: {len(codes_with_conflicts)/df_int_valids['barcode_norm'].nunique()*100:.2f}%")

print(f"\n DISTRIBUCIÓN DE CONFLICTOS:")
print(f"   Códigos con 2 marcas diferentes: {(codes_with_conflicts == 2).sum()}")
print(f"   Códigos con 3 marcas diferentes: {(codes_with_conflicts == 3).sum()}")
print(f"   Códigos con 4+ marcas diferentes: {(codes_with_conflicts >= 4).sum()}")


INFORME DE CALIDAD 
CONFLICTOS DE MARCAS POR CÓDIGO DE BARRAS

 ESTADÍSTICAS GENERALES:
   Total registros en Interfuerza: 73,451
   Total códigos de barras únicos: 73,303
   Códigos con múltiples marcas: 60
   Porcentaje del total: 0.08%

 DISTRIBUCIÓN DE CONFLICTOS:
   Códigos con 2 marcas diferentes: 60
   Códigos con 3 marcas diferentes: 0
   Códigos con 4+ marcas diferentes: 0


In [14]:
# ===========================================================================
# ANÁLISIS DE FORMATO PARA MARCAS MÚLTIPLES CORREGIDAS DE CÓDIGOS DUPLICADOS 
# ===========================================================================
print("="*63)
print("ANÁLISIS DE FORMATO PARA MARCAS MÚLTIPLES")
print("="*63)

# CORRECIÓN ORIGINAL
data_brands_path = os.path.join(notebook_dir, '..', 'data', 'raw', 'MarcasSeleccionadas.csv')
df_odoo_corrected = pd.read_csv(data_brands_path)
# iloc se utiliza para seleccionar un registro específico
print(df_odoo_corrected.iloc[3])
print(f"📋 Columnas: {list(df_odoo_corrected.columns)}")

"""
Filtro con str.contains para encontrar valores con en la columna de Marca
odoo_multiples_brands = df_odoo_corrected[df_odoo_corrected['Marca'].str.contains(",", na=False)]
"""
print("="*63)
# División de marcas múltiples para un mismo código
# Dividir valores en la columna Marca que tengan , utilizando Split y asignarlos a una nueva columna "Multiples_Brands"
df_odoo_corrected['Multiples_Brands'] = df_odoo_corrected['Marca'].str.split(",")
print("Productos con múltiples marcas:")
print(df_odoo_corrected.iloc[3])

ANÁLISIS DE FORMATO PARA MARCAS MÚLTIPLES
Código de barras          751214576491
Nombre              ETIQUET TSUM PAQ10
Marca                GRANMARK,DIVERTEX
Name: 3, dtype: object
📋 Columnas: ['Código de barras', 'Nombre', 'Marca']
Productos con múltiples marcas:
Código de barras            751214576491
Nombre                ETIQUET TSUM PAQ10
Marca                  GRANMARK,DIVERTEX
Multiples_Brands    [GRANMARK, DIVERTEX]
Name: 3, dtype: object


In [15]:
# =================================================================
# ANÁLISIS DE CORRECIÓN DE MARCAS MÚLTIPLES EN CÓDIGOS DUPLICADOS
# =================================================================
print("="*63)
print("ANÁLISIS DE CORRECIÓN DE MARCAS MÚLTIPLES EN CÓDIGOS DUPLICADOS")
print("="*63)

# CODIGOS CON CONFLICTO - INTERFUERZA 
# Identificar qué códigos se encuentran en codes_with_conflicts
df_int_conflicts = df_int_valids[df_int_valids['barcode_norm'].isin(codes_with_conflicts.index)]
# Convertir los valores de 'Marca' en String
df_int_conflicts.loc[:, 'Marca'] = df_int_conflicts['Marca'].astype(str)
# Dataframe con las columnas de código normalizado y marca. el .agg(', '.join) organiza varios valores de marca en una sola fila
df_int_grouped = df_int_conflicts.groupby('barcode_norm')['Marca'].agg(', '.join).reset_index()
print("CODIGOS INTERFUERZA")
print(df_int_grouped.head(5))

# CÓDIGOS CORREGIDOS EN ODOO
df_odoo_corrected['barcode_norm'] = df_odoo_corrected['Código de barras'].apply(barcode_normalization)
df_odoo_corrected.loc[:, 'Marca'] = df_odoo_corrected['Marca'].astype(str)
df_odoo_grouped = df_odoo_corrected.groupby('barcode_norm')['Marca'].agg(', '.join).reset_index()
print("="*66)
print("CODIGOS ODOO")
print(df_odoo_grouped.head(5))

# BASE INICIAL DE REPORTE
# Unión de dos DataFrames
df_report = df_int_grouped.merge(
    df_odoo_grouped, 
    on='barcode_norm', # Unión en código de barras
    how='inner', # Solo conserva códigos que existen en ambos DataFrames
    suffixes=('_interfuerza', '_odoo') # Agrega el sufijo a columnas con el mismo nombre
)

print("="*66)
print("REPORTE DE CORRECCIÓN DE MARCAS DUPLICADAS")
print(df_report.head())

ANÁLISIS DE CORRECIÓN DE MARCAS MÚLTIPLES EN CÓDIGOS DUPLICADOS
CODIGOS INTERFUERZA
  barcode_norm                                Marca
0  10100001258  STARMATE,POINTER, POINTER/METACOLOR
1  10200000786                          XX, OJENISA
2  10224000052                             X, RODEO
3       104142                       OJENISA, DAISY
4       104927                        MYLIN, SEDCIN
CODIGOS ODOO
  barcode_norm                                              Marca
0  10100001258  STARMATE/POINTER, METACOLOR,POINTER,STARMATE, ...
1  10200000786                                            OJENISA
2  10224000052                                                S/M
3       104142                                            OJENISA
4       104927                                             SEDCIN
REPORTE DE CORRECCIÓN DE MARCAS DUPLICADAS
  barcode_norm                    Marca_interfuerza  \
0  10100001258  STARMATE,POINTER, POINTER/METACOLOR   
1  10200000786                          XX

In [16]:
# ============================================
# ANÁLISIS COMPARATIVO
# ============================================
print("="*63)
print("ANÁLISIS DE COMPARATIVO DE CORRECCIONES")
print("="*63)

# COMPARA SI HAY AL MENOS UNA MARCA QUE COINCIDA
def compare_general_brands(df_report):
    int_brands = df_report['Marca_interfuerza']
    odoo_brands = df_report['Marca_odoo']
    
    # Manejo de nulos
    if pd.isna(int_brands) or pd.isna(odoo_brands):
        return 'No'
        
    # Conversión a string
    int_brands = str(int_brands)
    odoo_list = str(odoo_brands).split(',')
    
    for brand in odoo_list:
        if brand.strip() in int_brands:
            return 'Yes'
    return 'No'

# COMPARA SI TODAS LAS MARCAS COINCIDEN
def compare_specific_brands(df_report):
    int_brands = df_report['Marca_interfuerza']
    odoo_brands = df_report['Marca_odoo']

    if pd.isna(int_brands) or pd.isna(odoo_brands):
        return 'No'

    """
        Convertir a conjunto y separar las marcas en coma para compararlas directamente
        Los conjuntos no tienen orden y no permiten duplicados
        strip() elimina espacios alrededor de cada marca
    """
    int_set = set([b.strip() for b in str(int_brands).split(',')])
    odoo_set = set([b.strip() for b in str(odoo_brands).split(',')])
    
    # Verificar si TODAS las marcas de Odoo están en Interfuerza
    return 'Yes' if int_set == odoo_set else 'No'

# Aplicar las funciones para todas las filas
df_report['Comparación General'] = df_report.apply(compare_general_brands, axis=1)
df_report['Comparación Precisa'] = df_report.apply(compare_specific_brands, axis=1)
print(df_report.head(5))


ANÁLISIS DE COMPARATIVO DE CORRECCIONES
  barcode_norm                    Marca_interfuerza  \
0  10100001258  STARMATE,POINTER, POINTER/METACOLOR   
1  10200000786                          XX, OJENISA   
2  10224000052                             X, RODEO   
3       104142                       OJENISA, DAISY   
4       104927                        MYLIN, SEDCIN   

                                          Marca_odoo Comparación General  \
0  STARMATE/POINTER, METACOLOR,POINTER,STARMATE, ...                 Yes   
1                                            OJENISA                 Yes   
2                                                S/M                  No   
3                                            OJENISA                 Yes   
4                                             SEDCIN                 Yes   

  Comparación Precisa  
0                  No  
1                  No  
2                  No  
3                  No  
4                  No  


In [17]:
# ============================================
# REPORTE ESTADÍSTICO FINAL
# ============================================

print("="*60)
print("REPORTE DE RESOLUCIÓN DE CONFLICTOS")
print("="*60)

# TOTALES INTERFUERZA - ODOO
print(f"\n TOTAL DE CÓDIGOS CON CONFLICTOS: {len(codes_with_conflicts)}")

print(f"\n ANTES (Interfuerza):")
print(f"   Códigos con 2 marcas diferentes: {(codes_with_conflicts == 2).sum()}")
print(f"   Códigos con 3 marcas diferentes: {(codes_with_conflicts == 3).sum()}")
print(f"   Códigos con 4+ marcas diferentes: {(codes_with_conflicts >= 4).sum()}")

df_report['odoo_brand_count'] = df_report['Marca_odoo'].str.split(',').str.len()
print(f"\n DESPUÉS (Odoo):")
print(f" Códigos con 1 marca: {(df_report['odoo_brand_count'] == 1).sum()}")
print(f" Códigos con 2 marcas: {(df_report['odoo_brand_count'] == 2).sum()}")
print(f" Códigos con 3+ marcas: {(df_report['odoo_brand_count'] >= 3).sum()}")

# COMPARACIÓN DE MARCAS CORREGIDAS INTERFUERZA VS ODOO
print(f"\n MATCH DE MARCAS INTERFUERZA-ODOO")
print(f" Coincidencia en alguna marca: {(df_report['Comparación General'] == 'Yes').sum()} ({(df_report['Comparación General'] == 'Yes').sum()/len(df_report)*100:.1f}%)")
print(f" Concidencia en todas las marcas: {(df_report['Comparación Precisa'] == 'Yes').sum()} ({(df_report['Comparación Precisa'] == 'Yes').sum()/len(df_report)*100:.1f}%)")

resolved = ((df_report['Marca_interfuerza'].str.split(',').str.len()) > 1) & (df_report['odoo_brand_count'] == 1)
partial = (df_report['odoo_brand_count'] > 1)
no_coincidence = (df_report['Comparación General'] == 'No')

print("\n" + "="*60)
print("RESUMEN")
print("="*60)
print(f"""
Un total de {len(codes_with_conflicts)} códigos de barra estaban duplicados, resultando en multiple marcas que fueron identificados en Interfuerza.

ACCIONES TOMADAS:
- Cada uno de los {len(codes_with_conflicts)} casos fue manualmente revisado en Odoo.
- Muchos códigos eran duplicados con marcas erróneas (mal escritas, inexistentes, desfasadas, etc).
- Para productos diferentes con el mismo código se decició en base de la activadad del producto. 

RESULTADOS:
- ✅ {resolved.sum()} códigos ({resolved.sum()/len(df_report)*100:.1f}%) fueron simplificados a una sola marca
- ✅ {partial.sum()} códigos ({partial.sum()/len(df_report)*100:.1f}%) mantienen más de una marca
- ❌ {no_coincidence.sum()} códigos ({no_coincidence.sum()/len(df_report)*100:.1f}%) no tienen coincidencia con Interfuerza
""")

REPORTE DE RESOLUCIÓN DE CONFLICTOS

 TOTAL DE CÓDIGOS CON CONFLICTOS: 60

 ANTES (Interfuerza):
   Códigos con 2 marcas diferentes: 60
   Códigos con 3 marcas diferentes: 0
   Códigos con 4+ marcas diferentes: 0

 DESPUÉS (Odoo):
 Códigos con 1 marca: 50
 Códigos con 2 marcas: 8
 Códigos con 3+ marcas: 1

 MATCH DE MARCAS INTERFUERZA-ODOO
 Coincidencia en alguna marca: 44 (74.6%)
 Concidencia en todas las marcas: 5 (8.5%)

RESUMEN

Un total de 60 códigos de barra estaban duplicados, resultando en multiple marcas que fueron identificados en Interfuerza.

ACCIONES TOMADAS:
- Cada uno de los 60 casos fue manualmente revisado en Odoo.
- Muchos códigos eran duplicados con marcas erróneas (mal escritas, inexistentes, desfasadas, etc).
- Para productos diferentes con el mismo código se decició en base de la activadad del producto. 

RESULTADOS:
- ✅ 50 códigos (84.7%) fueron simplificados a una sola marca
- ✅ 9 códigos (15.3%) mantienen más de una marca
- ❌ 15 códigos (25.4%) no tienen coinci

In [27]:
# =====================================================================================
# ASIGNACIÓN DE MARCAS A PRODUCTOS SIN MARCA
# =====================================================================================

# Productos sin marca
df_without_brand = df_odoo[df_empty].copy()
# Excluir los que tienen código en codes_with_conflicts
df_without_brand = df_without_brand[~df_without_brand['barcode_norm'].isin(codes_with_conflicts.index)]

print(f"ANALISIS DE PRODUCTOS SIN MARCA")
print(f"Productos sin marca: {len(df_odoo[df_empty])}")
print(f"Productos sin marca excluyendo códigos corregidos: {len(df_without_brand)}")

# Aplicación de mapa de marcas de Interfuerza
df_without_brand['marca_asignada'] = df_without_brand['barcode_norm'].map(brands_map)

assigned = df_without_brand['marca_asignada'].notna().sum()
total_pendings = len(df_without_brand) - assigned
pendings = df_without_brand[df_without_brand['marca_asignada'].isna()]
print(f"✅ Resueltos automáticamente: {assigned} ({assigned/len(df_without_brand)*100:.1f}%)")
print(f"❌ Pendientes (sin coincidencia): {total_pendings} ({total_pendings/len(df_without_brand)*100:.1f}%)")

print("="*66)
print("PRODUCTOS SIN MARCA")
print("="*66)
print(pendings.iloc[500])

# Guardado de archivo con marcas asignadas listo para importación en Odoo
df_odoo_brands_import = df_without_brand[df_without_brand['marca_asignada'].notna()].copy()
df_odoo_brands_import = df_odoo_brands_import[['id', 'marca_asignada']]
df_odoo_brands_import.columns = ['id', 'x_studio_many2many_field_37q_1irl4uc58']

data_odoo_missing_brands_path = os.path.join(notebook_dir, '..', 'data', 'processed', 'MarcasParaActualizar.csv')
df_odoo_brands_import.to_csv(data_odoo_missing_brands_path, index=False)
print("="*66)
print("✅ Archivo 'MarcasParaActualizar.csv' generado")

ANALISIS DE PRODUCTOS SIN MARCA
Productos sin marca: 6954
Productos sin marca excluyendo códigos corregidos: 6954
✅ Resueltos automáticamente: 6136 (88.2%)
❌ Pendientes (sin coincidencia): 818 (11.8%)
PRODUCTOS SIN MARCA
id                                        __export__.product_template_219325_55d8d143
name                                                   MARCADOR FRANCO ACRILICO PLATA
barcode                                                                 7707759480284
default_code                                                                  FA41428
x_studio_many2many_field_37q_1irl4uc58                                            NaN
categ_id                                              Librería / Lápices y Bolígrafos
seller_ids                                                         PROV-METRIC PANAMA
barcode_norm                                                            7707759480284
marca_asignada                                                                    NaN
Name:

In [26]:
# =====================================================================================
# IDENTIFICACIÓN Y CREACIÓN DE MARCAS NO EXISTENTES EN ODOO
# =====================================================================================

assigned_brands = df_odoo_brands_import['x_studio_many2many_field_37q_1irl4uc58'].unique()
print(f"Cantidad de marcas únicas por asignar a productos sin marca en Odoo: {len(assigned_brands)}")

data_odoo_brands_path = os.path.join(notebook_dir, '..', 'data', 'raw', 'MarcasOdoo.csv')
df_odoo_brands = pd.read_csv(data_odoo_brands_path)
print(f"Marcas existentes en Odoo: {len(df_odoo_brands)}")
print("="*66)
print("MUESTRA DE MARCAS EN ODOO")
print("="*66)
print(df_odoo_brands.head())

# IDENTIFICACIÓN DE MARCAS FALTANTES EN ODOO
odoo_brands_standardized = set(
    df_odoo_brands['name'].astype(str).str.strip().str.lower()
)

# Encontrar marcas que no existen en Odoo
new_odoo_brands = []
for brand in assigned_brands:
    brand_str = str(brand).strip()
    if brand_str.lower() not in odoo_brands_standardized:
        new_odoo_brands.append(brand_str)

# Eliminar duplicados por si el mismo nombre aparece varias veces
new_odoo_brands = list(set(new_odoo_brands))

print("="*66)
print("MUESTRA DE MARCAS NUEVAS PARA ODOO")
print("="*66)
print(f"Marcas nuevas a crear: {len(new_odoo_brands)}")
print("-"*66)
print("Muestra de marcas")
print(new_odoo_brands[:5])
print("-"*66)

df_new_odoo_brands = pd.DataFrame({
    'name': new_odoo_brands
})

data_new_odoo_brands_path = os.path.join(notebook_dir, '..', 'data', 'processed', 'MarcasNuevasOdoo.csv')
df_new_odoo_brands.to_csv(data_new_odoo_brands_path, index=False)
print("Archivo 'MarcasNuevasOdoo.csv' generado")
print(f"Contiene {len(df_new_odoo_brands)} marcas para crear en Odoo")

Cantidad de marcas únicas por asignar a productos sin marca en Odoo: 946
Marcas existentes en Odoo: 1691
MUESTRA DE MARCAS EN ODOO
                                  id    name
0  __export__.product_tag_2_53e61365  Corsel
1  __export__.product_tag_3_a1f95b5d  Direct
2  __export__.product_tag_4_9ac42890    BODA
3  __export__.product_tag_5_b39c9e2f     S/M
4  __export__.product_tag_6_5cefcf1f     DFH
MUESTRA DE MARCAS NUEVAS PARA ODOO
Marcas nuevas a crear: 645
------------------------------------------------------------------
Muestra de marcas
['INVATEX', 'FURNITURE', 'TIFFANY', 'EASTER', 'MICRO LITHLUM CELL']
------------------------------------------------------------------
Archivo 'MarcasNuevasOdoo.csv' generado
Contiene 645 marcas para crear en Odoo
